# 4.1 · Testing AI Agents with Granite

Welcome to the introductory workshop on **Testing AI Agents** with IBM Granite.

Testing AI agents is analogous to Test-Driven Development (TDD) in traditional software engineering.
Because agents make autonomous decisions, choose tools, and generate natural language responses,
testing requires specialized techniques beyond standard unit testing:

- **Step 4. Minimal Function Calling Agent Setup** (`create_agent`)
- **Step 5. LLM-as-a-Judge (LLMaaJ)** for evaluating natural language final responses
- **Step 6. Programmatic Trajectory Evaluations** for verifying internal tool selection and parameter extraction
- **Step 7. Multi-Turn Evaluations** for contextual conversation memory and multi-step reasoning
- **Step 8. Non-Functional Requirements (NFR)** telemetry (latency & token tracking)

For deeper dives and reusable architectures, refer to the full recipes in the [Granite Agent Cookbook](https://github.com/ibm-granite-community/granite-agent-cookbook/tree/main/recipes):
- [Function Calling Agent Recipe](https://github.com/ibm-granite-community/granite-agent-cookbook/blob/main/recipes/Function_Calling/Function_Calling_Agent.ipynb)
- [Agent TDD Recipe](https://github.com/ibm-granite-community/granite-agent-cookbook/blob/main/recipes/Agent_TDD/Function_Calling_Agent_TDD.ipynb)
- [Non-Functional Requirements Recipe](https://github.com/ibm-granite-community/granite-agent-cookbook/blob/main/recipes/NFR_Agents/NFR_Agents.ipynb)
- [Testing Agents Guide](https://github.com/ibm-granite-community/granite-agent-cookbook/blob/main/testing_agents.md)

# Steps

## Step 1. Set up your environment

While you can choose from several tools, this recipe is best suited for a Jupyter Notebook. Jupyter Notebooks are widely used within data science to combine code with various data sources such as text, images and data visualizations. 

You can run this notebook in [Colab](https://colab.research.google.com/), or download it to your system and [run the notebook locally](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Getting_Started_with_Jupyter_Locally/Getting_Started_with_Jupyter_Locally.md). 

To avoid Python package dependency conflicts, we recommend setting up a [virtual environment](https://docs.python.org/3/library/venv.html).

Note, this notebook is compatible with Python 3.13 and well as Python 3.11 and 3.12. To check your python version, you can run the `!python --version` command in a code cell.


## Step 2. Set up a Granite AI model instance

This notebook requires IBM Granite models to be served by an AI model runtime so that the models can be invoked or called. This notebook can use a locally accessible [Ollama](https://ollama.com) server to serve the models, or the [Replicate](https://replicate.com) cloud service.

* See [Getting Started with Replicate](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Getting_Started/Getting_Started_with_Replicate.ipynb) for information on getting ready to use Replicate.
* See [Getting Started with Ollama](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Getting_Started/Getting_Started_with_Ollama.ipynb) for information on getting ready to use Ollama.

Prior to running this notebook, you should have either pulled the Granite model into a local Ollama server (`ollama pull granite4.2:3b`), or set up Replicate access and obtained an [API token](https://replicate.com/account/api-tokens). The `get_llm()` helper below uses Ollama when the model is available there, and Replicate otherwise.

## Step 3. Install relevant libraries and set up credentials and the Granite model

We'll need a few libraries for this recipe. We will be using LangGraph and LangChain libraries to use Granite.

In [ ]:
! echo "::group::Install Dependencies"
%pip install uv
! uv pip install "git+https://github.com/ibm-granite-community/utils.git" \
    langgraph \
    langchain \
    langchain_ollama \
    "langchain_replicate @ git+https://github.com/ibm-granite-community/langchain-replicate.git"
! echo "::endgroup::"

Now we will create the LangChain object to use the Granite model.

### Connect to Granite with `get_llm()`

In [ ]:
import os

import requests
from ibm_granite_community.notebook_utils import get_env_var


def ollama_has_model(host: str, model: str) -> bool:
    """True only if `host` answers AND `model` is already pulled there.

    Checking reachability alone is not enough: a running Ollama without the
    requested tag would otherwise trigger an on-demand pull, which needs
    network access we don't have on the day.
    """
    try:
        tags = requests.get(f"{host}/api/tags", timeout=2).json()
        return model in {m["name"] for m in tags.get("models", [])}
    except requests.RequestException:
        return False


def get_llm(
    model: str | None = None,
    *,
    backend: str | None = None,
    stop: list[str] | None = None,
    repetition_penalty: float | None = None,
    **kwargs,
):
    """Return a LangChain chat model for Granite.

    Args:
        model: An Ollama tag (e.g. "granite4.2:3b") or a Replicate model path
            (e.g. "ibm-granite/granite-4.2-8b"). Defaults to the `GRANITE_MODEL`
            env var (or "granite4.2:3b") for Ollama, and to `REPLICATE_MODEL`
            (or "ibm-granite/granite-4.2-8b") for Replicate.
        backend: Force "ollama" or "replicate"; omit to auto-resolve
            (Replicate if `REPLICATE_API_TOKEN` is set in the environment,
            else Ollama if the tag is pulled locally, Replicate otherwise).
        stop: Stop sequences, passed the way each backend expects them.
        repetition_penalty: Penalty for repeated tokens, passed the way each
            backend expects it.
        **kwargs: Extra keyword arguments passed through to the chat model.
    """
    host = get_env_var("OLLAMA_HOST", "http://127.0.0.1:11434")
    ollama_model = model or get_env_var("GRANITE_MODEL", "granite4.2:3b") or "granite4.2:3b"

    if backend is None:
        # A Replicate token in the environment is an explicit choice, so it wins over a local Ollama.
        if os.environ.get("REPLICATE_API_TOKEN"):
            backend = "replicate"
        elif ollama_has_model(host, ollama_model):
            backend = "ollama"

    if backend == "ollama":
        from langchain_ollama import ChatOllama

        return ChatOllama(
            model=ollama_model,
            base_url=host,
            num_predict=8192,  # Set the maximum number of tokens to generate as output.
            stop=stop,
            repeat_penalty=repetition_penalty,
            **kwargs,
        )

    from langchain_replicate import ChatReplicate

    # Only a Replicate model path ("owner/name") can be sent to Replicate; an Ollama tag cannot.
    if model and "/" in model:
        replicate_model = model
    else:
        replicate_model = get_env_var("REPLICATE_MODEL", "ibm-granite/granite-4.2-8b") or "ibm-granite/granite-4.2-8b"

    model_kwargs = {
        "max_completion_tokens": 4096,  # Set the maximum number of tokens to generate as output.
        "chat_template_kwargs": {"enable_thinking": False},
    }
    if stop:
        model_kwargs["stop"] = stop
    if repetition_penalty is not None:
        model_kwargs["repetition_penalty"] = repetition_penalty

    return ChatReplicate(
        model=replicate_model,
        replicate_api_token=get_env_var("REPLICATE_API_TOKEN"),
        model_kwargs=model_kwargs,
        **kwargs,
    )

In [ ]:
from langchain_core.utils.utils import convert_to_secret_str

llm = get_llm()
print(f"Connected via {type(llm).__name__}, model={llm.model!r}")

## Step 4. Minimal Function Calling Agent Setup

In this section, we set up our Granite model runtime and define two sample tools
(`get_current_weather` and `get_stock_price`) that return hard-coded responses, so the tests
are deterministic and need no external weather or stock API keys.

In [ ]:
import inspect
import os
import time

from langchain.agents import create_agent
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph.state import CompiledStateGraph

# Helper to retrieve environment variables safely
try:
    from ibm_granite_community.notebook_utils import get_env_var
except ImportError:
    def get_env_var(var_name: str, default: str = "") -> str:
        return os.environ.get(var_name, default)



In [ ]:
def get_current_weather(location: str) -> dict:
    """Fetches the current weather for a given city location.

    Args:
        location: The name of the city (e.g., 'Miami', 'London').

    Returns:
        A dictionary containing temperature in Celsius, description, and humidity.
    """
    # Hard-coded response so tests are deterministic
    return {
        "location": location,
        "description": "thunderstorms",
        "temperature": 25.3,
        "humidity": 94,
    }


def get_stock_price(ticker: str, date: str, currency: str = "USD") -> dict:
    """Retrieves the low and high stock prices for a given ticker and date.

    Args:
        ticker: The stock ticker symbol (e.g., 'IBM', 'AAPL').
        date: The date in 'YYYY-MM-DD' format.
        currency: Optional. Currency code for the prices (e.g., 'USD', 'EUR'). Defaults to 'USD'.

    Returns:
        A dictionary containing the low and high stock prices.
    """
    # Hard-coded response so tests are deterministic
    return {
        "ticker": ticker,
        "date": date,
        "currency": currency,
        "low": "245.4500",
        "high": "249.0300",
    }


# Instantiate the function-calling agent using prebuilt factory
tools = [get_current_weather, get_stock_price]
agent: CompiledStateGraph = create_agent(
    model=llm,
    tools=tools,
)

print("Agent initialized with tools:", [t.__name__ for t in tools])

## Step 5. LLM-as-a-Judge (LLMaaJ) for Final Response Evaluation

Traditional string matching (`in` or exact equality) is fragile when evaluating natural language
responses because models phrase answers in diverse, valid ways.

**LLM-as-a-Judge** uses an LLM to evaluate the final answer against:
1. The original user query.
2. The expected contents of the response.
3. An evaluation rubric (factual consistency, completeness, and lack of hallucinations).

**Test case shape (simplest):** a `name`, the `input` prompt, and the `expected_agent_response_contents`.
The agent is run on the prompt, and the judge decides whether its final answer contains the expected
information and returns PASS/FAIL.

In [ ]:
# LLMaaJ test case: name + input prompt + what the final response should contain
llmaaj_test = {
    "name": "weather_final_response",
    "input": "What is the weather in Miami?",
    "expected_agent_response_contents": "Miami has thunderstorms, a temperature of 25.3 degrees Celsius, and 94% humidity.",
}

### Step 5.1. Run the agent on the test case

Run the agent on the prompt from the test case and collect its final response.

In [ ]:
print(f"Running LLM-as-a-Judge test: {llmaaj_test['name']}")
agent_output = agent.invoke({"messages": [HumanMessage(content=llmaaj_test["input"])]})
agent_response = str(agent_output["messages"][-1].content)
print("Agent Response:\n", agent_response)

### Step 5.2. Build the judge prompt

Combine the user query, the expected response contents, the agent response and the rubric into a single prompt for the judge.

In [ ]:
judge_prompt = f"""You are an impartial AI evaluation judge. Evaluate whether the agent's response accurately and completely answers the user's query and contains the expected information.

[USER QUERY]
{llmaaj_test["input"]}

[EXPECTED RESPONSE CONTENTS]
{llmaaj_test["expected_agent_response_contents"]}

[AGENT RESPONSE]
{agent_response}

[RUBRIC]
- PASS if the response conveys all of the expected contents without contradictions or hallucinated facts.
- FAIL if the response misrepresents the expected contents, omits key information, or includes false claims.

Provide your evaluation in this format:
VERDICT: [PASS or FAIL]
REASONING: [1-2 sentences explaining your decision]
"""
print("\nJudge Prompt:\n", judge_prompt)

### Step 5.3. Invoke the judge

Invoke the judge and report success or failure.

In [ ]:
judge_result = llm.invoke([HumanMessage(content=judge_prompt)])
judge_output = judge_result.content if isinstance(judge_result.content, str) else str(judge_result.content)
judge_passed = "VERDICT: PASS" in judge_output.upper()

print("Judge Verdict:", "PASS" if judge_passed else "FAIL")
print("Judge Feedback:\n", judge_output.strip())

## Step 6. Programmatic Evaluation of Agent Internals (Trajectory Evals)

While LLM-as-a-Judge evaluates the final text output, **trajectory evaluation** checks *how*
the agent reached that conclusion. This is critical when agents call sensitive or state-changing tools.

**Test case shape (one step up):** same `name` and `input` as before, but instead of describing the
expected response we describe the `expected_tool_calls`. After running the agent we check:
- **Tool Name**: Was the correct tool selected?
- **Tool Parameters**: Were the extracted arguments correct?
- **Optional Parameters**: `get_stock_price` has an optional `currency` parameter (default `USD`).
  If the user doesn't ask for a currency, the agent may omit it *or* pass the default `USD` explicitly.
  To avoid false failures we validate parameters in two steps:
  1. Every *expected* parameter must be present in the actual args with the expected value.
  2. Any *additional* actual parameters must be optional parameters of the tool (i.e. have a default).

We run two test cases against the same tool to exercise both sides of that optionality.

In [ ]:
# Tool-calling test cases: name + input prompt + the tool call(s) we expect the agent to make
tool_calling_tests = [
    {
        # Optional `currency` is NOT requested; USD is the default, so the agent passing it or omitting it are both fine
        "name": "stock_price_without_optional_param_needed",
        "input": "What were the IBM stock prices on September 5, 2025?",
        "expected_tool_calls": [
            {"name": "get_stock_price", "args": {"ticker": "IBM", "date": "2025-09-05"}},
        ],
    },
    {
        # Optional `currency` IS requested, so it should be extracted into the args
        "name": "stock_price_with_optional_param",
        "input": "What were the IBM stock prices on September 5, 2025, in euros?",
        "expected_tool_calls": [
            {"name": "get_stock_price", "args": {"ticker": "IBM", "date": "2025-09-05", "currency": "EUR"}},
        ],
    },
]

tools_by_name = {t.__name__: t for t in tools}


def optional_param_names(tool_name: str) -> set[str]:
    """Names of the tool's optional parameters (those with a default value)."""
    signature = inspect.signature(tools_by_name[tool_name])
    optional_params = set()
    for param_name, param in signature.parameters.items():
        if param.default is not inspect.Parameter.empty:
            optional_params.add(param_name)
    return optional_params


tool_calling_results = []

for test in tool_calling_tests:
    print(f"\n--- Running tool-calling test: {test['name']} ---")

    # 1. Run the agent on the prompt from the test case
    start_time = time.time()
    agent_output = agent.invoke({"messages": [HumanMessage(content=test["input"])]})
    latency_ms = (time.time() - start_time) * 1000
    messages = agent_output["messages"]

    # 2. Collect the tool calls the agent made from the AI messages
    actual_tool_calls = []
    for message in messages:
        if isinstance(message, AIMessage):
            for tool_call in message.tool_calls:
                actual_tool_calls.append({"name": tool_call["name"], "args": tool_call["args"]})

    expected_tool_calls = test["expected_tool_calls"]
    print("Actual Tool Calls:  ", actual_tool_calls)
    print("Expected Tool Calls:", expected_tool_calls)
    print("Agent Response:\n", messages[-1].content)

    # 3. Was the right tool called? Every expected tool name must be among the actual tool names.
    actual_tool_names = []
    for actual in actual_tool_calls:
        actual_tool_names.append(actual["name"])

    tool_name_passed = True
    for expected in expected_tool_calls:
        if expected["name"] not in actual_tool_names:
            tool_name_passed = False  # expected tool was never called

    # 4. Were the right parameters passed?
    #    a) Every expected arg must be in the actual args of the matching tool call, with the same value.
    #    b) Any extra actual args are fine only if they are optional parameters of the tool.
    params_passed = True
    for expected in expected_tool_calls:
        for actual in actual_tool_calls:
            if actual["name"] != expected["name"]:
                continue

            for param_name, expected_value in expected["args"].items():
                if param_name not in actual["args"]:
                    params_passed = False  # expected param is missing
                elif actual["args"][param_name] != expected_value:
                    params_passed = False  # expected param has the wrong value

            optional_params = optional_param_names(actual["name"])
            for param_name in actual["args"]:
                if param_name not in expected["args"] and param_name not in optional_params:
                    params_passed = False  # extra param is not an optional parameter of the tool

    print("Correct tool called:      ", "PASS" if tool_name_passed else "FAIL")
    print("Correct parameters passed:", "PASS" if params_passed else "FAIL")

    tool_calling_results.append({
        "name": test["name"],
        "passed": tool_name_passed and params_passed,
        "latency_ms": latency_ms,
    })

## Step 7. Multi-Turn / Multi-Step Agent Evaluation

Real-world agent interactions span multiple conversational turns.
In this section, we test whether the agent retains context across turns (e.g. resolving
*"How about in Tokyo?"* into a weather query for Tokyo based on prior turn context).

**Test case shape (most complex):** the same `name` as before, but `input` and `expected_tool_calls`
now live in a list of `turns`. Each turn has exactly the shape of the tool-calling test case from
Step 6 and is checked with the same tool name and parameter logic. The difference is that the
conversation history is carried between turns, so a later turn can depend on an earlier one.

### Going further: simulated users with hidden context

The turns above are scripted: every user message is fixed in advance. In some multi-turn scenarios you
will want something more realistic, a **simulated user** (typically another LLM) that is given a goal
plus some **hidden context** the agent does not know up front, such as the ticker symbol, the date or
the currency it actually cares about. The simulated user starts with a vague request
(e.g. *"What was the stock price last week?"*) and only **drip-feeds** the hidden details back when the
agent asks for them.

This tests whether the agent can:
- **Identify missing information** instead of guessing or hallucinating values for required parameters.
- **Ask the right clarifying questions** rather than calling a tool with incomplete arguments.
- **Use the answers** to eventually make the correct tool call with the correct parameters.

The same trajectory checks still apply: once the conversation ends, compare the tool calls the agent
made against the `expected_tool_calls` derived from the hidden context. You can also assert that no tool
call was made *before* the missing information was provided.

In [ ]:
# Multi-turn test case: name + a list of turns, each turn shaped like a tool-calling test case
multi_turn_test = {
    "name": "weather_follow_up_conversation",
    "turns": [
        {
            "input": "What's the weather like in London?",
            "expected_tool_calls": [
                {"name": "get_current_weather", "args": {"location": "London"}},
            ],
        },
        {
            # "How about in Tokyo?" only makes sense given the previous turn
            "input": "How about in Tokyo?",
            "expected_tool_calls": [
                {"name": "get_current_weather", "args": {"location": "Tokyo"}},
            ],
        },
    ],
}

print(f"Running multi-turn test: {multi_turn_test['name']}")
conversation_messages = []
multi_turn_results = []

for turn_number, turn in enumerate(multi_turn_test["turns"], start=1):
    print(f"\n--- Turn {turn_number}: {turn['input']} ---")

    # 1. Add this turn's input to the conversation and run the agent with the full history so far
    conversation_messages.append(HumanMessage(content=turn["input"]))
    history_length = len(conversation_messages)

    start_time = time.time()
    agent_output = agent.invoke({"messages": conversation_messages})
    latency_ms = (time.time() - start_time) * 1000
    conversation_messages = agent_output["messages"]

    # 2. Collect only the tool calls made during this turn (skip messages from earlier turns)
    actual_tool_calls = []
    for message in conversation_messages[history_length:]:
        if isinstance(message, AIMessage):
            for tool_call in message.tool_calls:
                actual_tool_calls.append({"name": tool_call["name"], "args": tool_call["args"]})

    expected_tool_calls = turn["expected_tool_calls"]
    print("Actual Tool Calls:  ", actual_tool_calls)
    print("Expected Tool Calls:", expected_tool_calls)
    print("Agent Response:\n", conversation_messages[-1].content)

    # 3. Was the right tool called? Every expected tool name must be among the actual tool names.
    actual_tool_names = []
    for actual in actual_tool_calls:
        actual_tool_names.append(actual["name"])

    tool_name_passed = True
    for expected in expected_tool_calls:
        if expected["name"] not in actual_tool_names:
            tool_name_passed = False  # expected tool was never called

    # 4. Were the right parameters passed? (same logic as the tool-calling test)
    #    a) Every expected arg must be in the actual args of the matching tool call, with the same value.
    #    b) Any extra actual args are fine only if they are optional parameters of the tool.
    params_passed = True
    for expected in expected_tool_calls:
        for actual in actual_tool_calls:
            if actual["name"] != expected["name"]:
                continue

            for param_name, expected_value in expected["args"].items():
                if param_name not in actual["args"]:
                    params_passed = False  # expected param is missing
                elif actual["args"][param_name] != expected_value:
                    params_passed = False  # expected param has the wrong value

            optional_params = optional_param_names(actual["name"])
            for param_name in actual["args"]:
                if param_name not in expected["args"] and param_name not in optional_params:
                    params_passed = False  # extra param is not an optional parameter of the tool

    print("Correct tool called:      ", "PASS" if tool_name_passed else "FAIL")
    print("Correct parameters passed:", "PASS" if params_passed else "FAIL")

    multi_turn_results.append({
        "turn": turn_number,
        "passed": tool_name_passed and params_passed,
        "latency_ms": latency_ms,
    })

## Step 8. Non-Functional Requirements (NFR) Telemetry & Summary Reporting

Functional correctness is not enough to productionise an agent. **Non-functional requirements (NFRs)**
determine whether an agent is practical to run at scale, and they should be measured and tracked
alongside pass/fail results:

- **Efficiency:** How many extra steps and tool calls did the agent use to reach the answer? An agent
  that makes redundant or repeated tool calls, or loops before answering, may still pass functionally
  while being wasteful. Compare the number of steps and tool calls against the minimum needed
  (e.g. the length of `expected_tool_calls`).
- **Latency:** How long did the agent take to return a response? Autonomous agent loops compound latency
  across LLM calls, tool calls and conversation turns, which directly affects user experience.
- **Cost:** How much did the run cost? Cost scales with the tokens consumed by every LLM call, and
  longer agent loops and growing multi-turn history increase it further.

Below, we compute and display high-level telemetry and test pass rates. For this workshop we report
latency, which we have measured for every test. Efficiency and cost need extra data from the agent
runs, which the NFR recipe linked at the top covers in more depth.

In [ ]:
st_passed = sum(1 for r in tool_calling_results if r["passed"])
st_total = len(tool_calling_results)
mt_passed = sum(1 for r in multi_turn_results if r["passed"])
mt_total = len(multi_turn_results)

total_passed = st_passed + mt_passed
total_tests = st_total + mt_total

all_latencies = [r["latency_ms"] for r in tool_calling_results] + [r["latency_ms"] for r in multi_turn_results]
avg_latency = sum(all_latencies) / len(all_latencies) if all_latencies else 0

print("\n" + "=" * 55)
print("             AGENT TEST SUITE SUMMARY")
print("=" * 55)
print(f"Tool-Calling Pass Rate: {st_passed}/{st_total} ({(st_passed/st_total)*100:.1f}%)")
print(f"Multi-Turn Pass Rate  : {mt_passed}/{mt_total} ({(mt_passed/mt_total)*100:.1f}%)")
print(f"Overall Pass Rate     : {total_passed}/{total_tests} ({(total_passed/total_tests)*100:.1f}%)")
print(f"Average Latency       : {avg_latency:.2f} ms")
print("=" * 55)
print("\nWorkshop completed. See the Testing Agents Guide for advanced evaluation strategies:")
print("https://github.com/ibm-granite-community/granite-agent-cookbook/blob/main/testing_agents.md")